## Downloading Necessary Libraries & Packages

In [ ]:
!pip install open3d

## Importing Necessary Libraries, Packages and Functions

In [1]:
import open3d as o3d
import numpy as np
import nibabel as nib
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import map_coordinates

from CEHE_Algorithm import get_density_map, CEHE, inverted_average_fn
from test_aug_mat import test_edge_aug_matrix_1, test_edge_aug_matrix_2

## Some Utility Functions:
These functions should be implemented in AOMT and the algorithm implemented here should be supplied by vertices coordinates of the mesh, faces of the mesh, and density of each vertex.

In [2]:
def build_density_map(file_path):
  img = nib.load(file_path)
  img_arr = img.get_fdata()
  normalized_img_arr = (img_arr - np.mean(img_arr)) / np.std(img_arr)
  enhanced_img_arr = CEHE(normalized_img_arr, inverted_average_fn, 3, 65536)
  enhanced_image = nib.Nifti1Image(enhanced_img_arr, affine=np.eye(4))
  return get_density_map(enhanced_image, 1)

In [3]:
def read_off_file(file_path):
  mesh = o3d.io.read_triangle_mesh(file_path)
  vertices = np.asarray(mesh.vertices)
  faces = np.asarray(mesh.triangles)
  return vertices, np.sort(faces, axis=1)

In [4]:
d_map = build_density_map("/content/BraTS2021_00000_flair.nii.gz")

In [5]:
vertices, faces = read_off_file('/content/BraTS2021_Training_00000_flair.off')

## A Function to compute vertices density
It performs Trilinear Interpolation on each vertex coordinate to get its density from the image density map.

In [6]:
get_vertices_density = lambda d_map, vertices: map_coordinates(d_map, vertices.T, order=1, mode='nearest')

## Building Edge Augmented Matrix

Edge Augmented Matrix is a **(#edges x 8) matrix**, its columns are as follows: </br>
1. **Columns {0, 1}:** Indices of the first and second vertices of the edge.
2. **Columns {2, 3}:** Indices of the 2 vertices such that if each is combined with the edge, a triangle in the mesh is formed.
3. **Columns {4, 5}:** Areas of the Two Triangles formed by the vertices in the previous 2 columns and the edge respectively.
4. **Columns {6, 7}:** Densities of the Two Triangles formed by the vertices in the previous 2 columns and the edge respectively.

### a Function to build Adjacency Matrix for the Mesh

In [7]:
def build_adj_matrix(mesh_faces, num_vertices):
    adj_matrix = np.zeros((num_vertices, num_vertices), dtype=bool)

    def fill_adj_matrix(face):
      v1, v2, v3 = face[0], face[1], face[2]
      adj_matrix[v1, v2], adj_matrix[v1, v3], adj_matrix[v2, v3] = True, True, True
      adj_matrix[v2, v1], adj_matrix[v3, v1], adj_matrix[v3, v2] = True, True, True

    np.apply_along_axis(fill_adj_matrix, axis=1, arr=mesh_faces)
    return adj_matrix

### a Function to build the first 4 comlums in Edge Augmented Matrix

In [8]:
def build_edge_aug_matrix_f4(adj_matrix, mesh_faces):
  edge_aug_matrix = []
  row_indices, col_indices = np.where(adj_matrix)
  edges = np.vstack([row_indices, col_indices]).T
  edges = edges[edges[:, 0] < edges[:, 1]]

  def get_triangle_vertices(edge):
    common_vertices = adj_matrix[edge[0]] & adj_matrix[edge[1]]
    if np.sum(common_vertices) > 1: # we have two triangles sharing this edge
      possible_face_heads = np.where(common_vertices)[0]
      e0_repeat = np.full_like(possible_face_heads, edge[0])
      e1_repeat = np.full_like(possible_face_heads, edge[1])

      possible_faces = np.column_stack((e0_repeat, e1_repeat, possible_face_heads))
      possible_faces = np.sort(possible_faces, axis=1)
      face_indices = np.where(np.all(possible_faces[:, None, :] == mesh_faces, axis=-1))[0]

      if(len(face_indices) > 1):
        x = np.array(np.hstack([edge, possible_face_heads[face_indices]])[:])
        edge_aug_matrix.append(x)

  np.apply_along_axis(get_triangle_vertices, axis=1, arr=edges)
  return np.array(edge_aug_matrix)

### a Function to compute columns 4 and 5 in Edge Augmented Matrix

In [9]:
compute_face_area = lambda v1, v2, v3: 0.5 * np.linalg.norm(np.cross(v2 - v1, v3 - v1))

In [10]:
def build_edge_aug_matrix_4_5(edge_aug_matrix, vertices):
  def compute_entry_4_5(row):
    v1, v2 = vertices[int(row[0])], vertices[int(row[1])]
    v3, v4 = vertices[int(row[2])], vertices[int(row[3])]
    row[4:6] = np.array([compute_face_area(v1, v2, v3), compute_face_area(v1, v2, v4)])

  np.apply_along_axis(compute_entry_4_5, axis=1, arr=edge_aug_matrix)
  return edge_aug_matrix[:, 4:6]

### a Function to compute columns 6 and 7 in Edge Augmented Matrix

In [11]:
def build_edge_aug_matrix_6_7(edge_aug_matrix, vertices_densities):
  def compute_entry_6_7(row):
    d_v1, d_v2 = vertices_densities[int(row[0])], vertices_densities[int(row[1])]
    d_v3, d_v4 = vertices_densities[int(row[2])], vertices_densities[int(row[3])]
    row[6:8] = np.array([(d_v1 + d_v2 + d_v3) / 3, (d_v1 + d_v2 + d_v4) / 3])

  np.apply_along_axis(compute_entry_6_7, axis=1, arr=edge_aug_matrix)
  return edge_aug_matrix[:, 6:8]

### a Facade Function to build Edge Augmented Matrix

In [12]:
def build_edge_aug_matrix(adj_matrix, vertices, faces, vertices_densities):
  edge_aug_matrix_mini = build_edge_aug_matrix_f4(adj_matrix, faces)
  edge_aug_matrix = np.zeros((edge_aug_matrix_mini.shape[0], 8))
  edge_aug_matrix[:, :4] = edge_aug_matrix_mini.reshape((edge_aug_matrix_mini.shape[0], 4))
  edge_aug_matrix[:, 4:6] = build_edge_aug_matrix_4_5(edge_aug_matrix, vertices)
  edge_aug_matrix[:, 6:8] = build_edge_aug_matrix_6_7(edge_aug_matrix, vertices_densities)
  return edge_aug_matrix

## ASEM Algorithm Main Function

In [13]:
def ASEM(mesh_vertices, mesh_faces, image_density_map):
  vertices_densities = get_vertices_density(image_density_map, vertices)
  adj_matrix = build_adj_matrix(mesh_faces, mesh_vertices.shape[0])
  edge_aug_matrix = build_edge_aug_matrix(adj_matrix, mesh_vertices, mesh_faces, vertices_densities)



## Tests:

In [14]:
adj_matrix = build_adj_matrix(faces, vertices.shape[0])
edge_aug_matrix = build_edge_aug_matrix(adj_matrix, vertices, faces, get_vertices_density(d_map, vertices))

In [16]:
test_edge_aug_matrix_2(edge_aug_matrix[78], vertices, get_vertices_density(d_map, vertices))

Areas and Densities are Correct
 Success!


### 1. Testing if first columns of Edge Augmented Matrix formed correctly

In [17]:
test_edge_aug_matrix_1(build_adj_matrix, build_edge_aug_matrix_f4)

[[False  True False False False  True  True]
 [ True False  True False False False  True]
 [False  True False  True False False  True]
 [False False  True False  True  True  True]
 [False False False  True False  True  True]
 [ True False False  True  True False  True]
 [ True  True  True  True  True  True False]]
Adjacency Matrix is correct
[[0 6 1 5]
 [1 6 0 2]
 [2 6 1 3]
 [3 4 5 6]
 [3 6 2 4]
 [4 5 3 6]
 [4 6 3 5]
 [5 6 0 4]]
First 4 Columns of Edge Augmented Matrix are correct
 Success!
